In [7]:
import argparse
import sys
from pathlib import Path

import torch
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from crop_embed.models.fp_head_model import (
    MLPModel, LinearModel, FPSumHeadModel,
)
from crop_embed import FixedWindowEmbedder, MetricLogger, metrics_path_for
from crop_embed.data.loading import prepare_data
from crop_embed.train import masked_mse, _compute_metrics


In [ ]:

# ── CLI ───────────────────────────────────────────────────────────────────────

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


# ── Dataset, targets, and the shared train/val split ──────────────────────────
# Generate the split first with: python scripts/cache_split.py --output <SPLIT_PATH>

SPLIT_PATH = "../splits/sativas413_seed42.pt"
data = prepare_data(split_path=SPLIT_PATH)
dataset    = data["dataset"]
Y          = data["Y"]
trait_cols = data["trait_cols"]
train_idx  = data["train_idx"]
val_idx    = data["val_idx"]


In [10]:

# ── 1. Load fixed window cache ────────────────────────────────────────────────
cache = "../checkpoints/sativas413_embeddings.ckpt.pt"
print(f"\nLoading cache from {cache} …")
embedder = FixedWindowEmbedder.from_file(cache, dataset)
cache           = embedder.cache.float()       # (n_fps, D)
sample_fp_index = embedder.sample_fp_index      # (n_samples, n_windows)
emb_dim  = cache.shape[1]
n_traits = Y.shape[1]

# The cache's sample→fingerprint index must line up with the dataset the split
# was built from; otherwise train_idx/val_idx point at the wrong samples.
if (sample_fp_index.shape != dataset.sample_fp_index.shape
        or not torch.equal(sample_fp_index, dataset.sample_fp_index)):
    raise SystemExit(
        "Cache sample→fingerprint index doesn't match the dataset built from the "
        "split's VCF/windowing. Regenerate the cache for this windowing."
    )
print(f"  {cache.shape[0]:,} fingerprints × {emb_dim} dims; {sample_fp_index.shape[0]} samples")

# ── 1.a Pre-sum into one embedding per sample ─────────────────────────────────
# Frozen cache → the per-sample sum is constant, so compute it once. embedding_bag
# fuses the gather+sum so the (n_samples, n_windows, D) intermediate never exists.

summed = F.embedding_bag(sample_fp_index, cache, mode="sum")   # (n_samples, D)
meaned = F.embedding_bag(sample_fp_index, cache, mode="mean")   # (n_samples, D)

train_ds = TensorDataset(summed[train_idx], Y[train_idx])
val_x = summed[val_idx].to(device)
val_y = Y[val_idx].to(device)
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)


Loading cache from ../checkpoints/sativas413_embeddings.ckpt.pt …
  53,398 fingerprints × 768 dims; 383 samples


In [12]:
print(meaned.norm(dim=0))

tensor([8.6161e-01, 2.0212e+00, 1.0580e+00, 1.4172e+00, 1.7094e+00, 8.7480e-01,
        1.4461e+00, 2.1239e+00, 1.2869e-01, 5.2673e-01, 2.5503e+00, 5.6803e-01,
        2.3406e-01, 1.2037e+00, 1.3640e-01, 4.3751e-01, 1.4877e+00, 1.5446e+00,
        3.6684e-01, 1.6845e+00, 1.2760e+00, 1.2783e+00, 3.9664e-02, 1.0412e-01,
        1.6159e+00, 9.4048e-01, 5.9869e+00, 1.1487e-01, 1.1989e+00, 8.8521e-01,
        5.8411e-01, 1.9868e+00, 2.1696e+00, 5.3390e-01, 1.1786e+00, 2.8809e-01,
        1.5969e+00, 1.1717e+00, 6.4843e-01, 7.1125e-01, 1.8917e+00, 8.3708e-02,
        4.5629e-02, 7.7265e-01, 2.9909e-01, 1.7733e+00, 1.9185e+00, 9.4887e-01,
        1.1963e+00, 2.6771e+00, 5.4429e-01, 1.4736e+00, 1.1004e+00, 9.5532e-01,
        5.0229e-01, 2.1038e+00, 1.9219e+00, 1.2741e+00, 1.9994e+00, 1.5279e-01,
        8.3980e-02, 3.4017e-01, 3.0103e-01, 2.6697e+00, 1.5260e+00, 7.1278e-01,
        1.1999e+00, 7.0090e-01, 3.0770e-01, 9.6723e-01, 1.4183e+00, 9.7966e-01,
        1.7749e+00, 5.8001e-01, 1.7983e+